In [1]:
import pandas as pd

df = pd.read_csv('Grocery_Inventory.csv', encoding='utf-8-sig')
display(df.head())
df.info()

,Product_Name,Catagory,Supplier_Name,Warehouse_Location,Status,Product_ID,Supplier_ID,Date_Received,Last_Order_Date,Expiration_Date,Stock_Quantity,Reorder_Level,Reorder_Quantity,Unit_Price,Sales_Volume,Inventory_Turnover_Rate,percentage
0,Bell Pepper,Fruits & Vegetables,Eimbee,20 Pennsylvania Parkway,Discontinued,29-017-6255,43-348-2450,3/1/2024,1/6/2025,1/31/2025,46,64,17,$4.60,96,55,1.96%
1,Vegetable Oil,Oils & Fats,Digitube,03643 Oakridge Lane,Backordered,79-569-8856,04-854-7165,4/1/2024,5/19/2024,6/11/2024,51,87,86,$2.00,24,83,0.91%
2,Parmesan Cheese,Dairy,BlogXS,73 Graedel Street,Discontinued,28-146-2641,82-995-0739,4/1/2024,12/21/2024,4/8/2024,38,67,66,$12.00,35,24,1.36%
3,Carrot,Fruits & Vegetables,Avaveo,44801 Myrtle Center,Discontinued,11-581-9869,22-867-3079,5/1/2024,12/12/2024,9/26/2024,51,60,98,$1.50,44,95,1.36%
4,Garlic,Fruits & Vegetables,Katz,6195 Monterey Center,Discontinued,13-202-4809,24-281-7685,5/1/2024,7/28/2024,5/20/2024,27,22,89,$7.00,91,77,2.17%


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 990 entries, 0 to 989
Data columns (total 17 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   Product_Name             990 non-null    object
 1   Catagory                 989 non-null    object
 2   Supplier_Name            990 non-null    object
 3   Warehouse_Location       990 non-null    object
 4   Status                   990 non-null    object
 5   Product_ID               990 non-null    object
 6   Supplier_ID              990 non-null    object
 7   Date_Received            990 non-null    object
 8   Last_Order_Date          990 non-null    object
 9   Expiration_Date          990 non-null    object
 10  Stock_Quantity           990 non-null    int64 
 11  Reorder_Level            990 non-null    int64 
 12  Reorder_Quantity         990 non-null    int64 
 13  Unit_Price               990 non-null    object
 14  Sales_Volume             990 non-null    i

# Category

In [2]:
categories = df[['Catagory']].dropna()
unique_categories = categories.drop_duplicates().reset_index(drop=True)
unique_categories['Catagory'] = unique_categories['Catagory'].str.strip().str.title()
unique_categories = unique_categories.rename(columns={'Catagory': 'category_name'})
display(unique_categories.head())

,category_name
0,Fruits & Vegetables
1,Oils & Fats
2,Dairy
3,Grains & Pulses
4,Seafood


In [4]:
from pathlib import Path
current_dir = Path.cwd()
CLEAN_DIR = current_dir.parent / "clean"
CLEAN_DIR.mkdir(parents=True, exist_ok=True)
file_path = CLEAN_DIR / "category_seed.json"
unique_categories.to_json(file_path, orient='records', force_ascii=False, indent=4)
print("Done")

Done


# Supplier

In [6]:
suppliers = df[['Supplier_Name']].dropna()
unique_suppliers = suppliers.drop_duplicates().reset_index(drop=True)
unique_suppliers = unique_suppliers.rename(columns={'Supplier_Name': 'supplier_name'})
unique_suppliers['supplier_name'] = unique_suppliers['supplier_name'].str.strip().str.title()
unique_suppliers['status'] = 'active'
display(unique_suppliers.head())

,supplier_name,status
0,Eimbee,active
1,Digitube,active
2,Blogxs,active
3,Avaveo,active
4,Katz,active


In [ ]:
from pathlib import Path
current_dir = Path.cwd()
CLEAN_DIR = current_dir.parent / "clean"
CLEAN_DIR.mkdir(parents=True, exist_ok=True)
file_path = CLEAN_DIR / "suppliers_seed.json"
unique_suppliers.to_json(file_path, orient='records', force_ascii=False, indent=4)
print(f"Done")

Done


# Product

In [9]:
import pandas as pd
import numpy as np

df_products = df[[
    'Product_Name', 'Catagory', 'Product_ID', 'Warehouse_Location', 
    'Stock_Quantity', 'Sales_Volume', 'Reorder_Level', 'Unit_Price', 'Expiration_Date'
]].copy()

df_products = df_products.rename(columns={
    'Product_Name': 'product_name',
    'Catagory': 'category_name',
    'Product_ID': 'sku_code',
    'Warehouse_Location': 'warehouse_location',
    'Stock_Quantity': 'stock_quantity',
    'Sales_Volume': 'sales_volume',
    'Reorder_Level': 'reorder_level',
    'Unit_Price': 'last_unit_price',
    'Expiration_Date': 'expiry_date'
})

if df_products['last_unit_price'].dtype == 'object':
    df_products['last_unit_price'] = df_products['last_unit_price'].str.replace(r'[^\d.]', '', regex=True)
df_products['last_unit_price'] = pd.to_numeric(df_products['last_unit_price'], errors='coerce').fillna(0)
df_products['currency'] = 'AUD'
df_products['status'] = 'in_stock'
df_products['expiry_date'] = pd.to_datetime(df_products['expiry_date']).dt.strftime('%Y-%m-%d')

display(df_products.head())

,product_name,category_name,sku_code,warehouse_location,stock_quantity,sales_volume,reorder_level,last_unit_price,expiry_date,currency,status
0,Bell Pepper,Fruits & Vegetables,29-017-6255,20 Pennsylvania Parkway,46,96,64,4.6,2025-01-31,AUD,in_stock
1,Vegetable Oil,Oils & Fats,79-569-8856,03643 Oakridge Lane,51,24,87,2.0,2024-06-11,AUD,in_stock
2,Parmesan Cheese,Dairy,28-146-2641,73 Graedel Street,38,35,67,12.0,2024-04-08,AUD,in_stock
3,Carrot,Fruits & Vegetables,11-581-9869,44801 Myrtle Center,51,44,60,1.5,2024-09-26,AUD,in_stock
4,Garlic,Fruits & Vegetables,13-202-4809,6195 Monterey Center,27,91,22,7.0,2024-05-20,AUD,in_stock


In [ ]:
from pathlib import Path
current_dir = Path.cwd()
CLEAN_DIR = current_dir.parent / "clean"
CLEAN_DIR.mkdir(parents=True, exist_ok=True)
file_path = CLEAN_DIR / "products_seed.json"
df_products.to_json(file_path, orient='records', force_ascii=False, indent=4)
print(f"Done")

Done
